# Fake Profile Detection Prototype
## Federated Multi-Modal System: Tabular + NLP + Vision → Meta-Learner → FastAPI
### Sections:
1. Install Dependencies
2. Data Loading & Preprocessing
3. Phase 1A: Tabular Model (RF/XGB/LR) — Federated Simulation
4. Phase 1B: NLP Model (GRU) — Federated Simulation
5. Phase 1C: Vision Model (ResNet18) — Federated Simulation
6. Federated Aggregator
7. Phase 2: Meta-Learner (Stacking)
8. Phase 3: FastAPI Server
9. Test the API

## Section 1: Install Dependencies

In [17]:
import sys
!{sys.executable} -m pip install -q pandas numpy scikit-learn xgboost torch torchvision transformers fastapi uvicorn requests Pillow aiohttp nest-asyncio pyngrok joblib

## Section 2: Data Loading & Preprocessing

In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# ── Load datasets ──────────────────────────────────────────────
df_profile = pd.read_csv('fake_profile_twitter.csv')
df_activity = pd.read_csv('raw_user_activities 1(in).csv')

print('Profile shape:', df_profile.shape)
print('Activity shape:', df_activity.shape)

Profile shape: (3351, 38)
Activity shape: (131284, 24)


In [20]:
# ── Profile dataset: derive fake label from temporal/behavioral heuristics ──
df_profile['created_at'] = pd.to_datetime(df_profile['created_at'], errors='coerce', utc=True)
df_profile['updated']    = pd.to_datetime(df_profile['updated'],    errors='coerce', utc=True)
df_profile['account_age_days'] = (df_profile['updated'] - df_profile['created_at']).dt.days.fillna(0)

# Heuristic fake label: low followers, high friends, default profile, no description, young account
df_profile['followers_count']  = pd.to_numeric(df_profile['followers_count'],  errors='coerce').fillna(0)
df_profile['friends_count']    = pd.to_numeric(df_profile['friends_count'],    errors='coerce').fillna(0)
df_profile['statuses_count']   = pd.to_numeric(df_profile['statuses_count'],   errors='coerce').fillna(0)
df_profile['favourites_count'] = pd.to_numeric(df_profile['favourites_count'], errors='coerce').fillna(0)
df_profile['listed_count']     = pd.to_numeric(df_profile['listed_count'],     errors='coerce').fillna(0)

df_profile['ff_ratio']         = df_profile['followers_count'] / (df_profile['friends_count'] + 1)
df_profile['has_description']  = df_profile['description'].notna().astype(int)
df_profile['default_profile']  = pd.to_numeric(df_profile['default_profile'], errors='coerce').fillna(1)

# Score-based heuristic (higher = more likely fake)
fake_score = (
    (df_profile['ff_ratio'] < 0.1).astype(int) +
    (df_profile['statuses_count'] < 10).astype(int) +
    (df_profile['default_profile'] == 1).astype(int) +
    (df_profile['has_description'] == 0).astype(int) +
    (df_profile['account_age_days'] < 30).astype(int) +
    (df_profile['followers_count'] < 5).astype(int)
)
df_profile['is_fake'] = (fake_score >= 3).astype(int)
print('Profile fake distribution:', df_profile['is_fake'].value_counts().to_dict())

Profile fake distribution: {0: 2111, 1: 1240}


In [19]:
# ── Activity dataset: prepare NLP + tabular features ──────────
df_activity['is_fake'] = df_activity['is_fake'].map({True: 1, False: 0, 'TRUE': 1, 'FALSE': 0}).fillna(0).astype(int)
df_activity['timestamp'] = pd.to_datetime(df_activity['timestamp'], errors='coerce')
df_activity['content']   = df_activity['content'].fillna('')

# Tabular features from activity
tab_act_cols = ['likes','comments','shares','hour_of_day','day_of_week','is_weekend',
                'character_count','hashtag_count','mention_count','contains_url']
df_activity[tab_act_cols] = df_activity[tab_act_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

# Encode categorical
for col in ['device','platform','media_type','content_type','language']:
    df_activity[col] = LabelEncoder().fit_transform(df_activity[col].astype(str))

print('Activity fake distribution:', df_activity['is_fake'].value_counts().to_dict())

Activity fake distribution: {0: 101439, 1: 29845}


## Section 3: Phase 1A — Tabular Model (RF / XGB / LR) with Federated Simulation

In [5]:
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import joblib, os

os.makedirs('saved_models', exist_ok=True)

# ── Features for tabular model ─────────────────────────────────
tab_profile_cols = ['statuses_count','followers_count','friends_count','favourites_count',
                    'listed_count','ff_ratio','has_description','default_profile','account_age_days']

X_tab = df_profile[tab_profile_cols].fillna(0).values
y_tab = df_profile['is_fake'].values

scaler_tab = StandardScaler()
X_tab_scaled = scaler_tab.fit_transform(X_tab)
X_tr, X_te, y_tr, y_te = train_test_split(X_tab_scaled, y_tab, test_size=0.2, random_state=42)

# ── Federated simulation: 2 client nodes each train on half the data ──
def federated_tabular_train(X_tr, y_tr):
    mid = len(X_tr) // 2
    clients = [
        {'X': X_tr[:mid],  'y': y_tr[:mid]},
        {'X': X_tr[mid:],  'y': y_tr[mid:]},
    ]
    # Each client trains RF + XGB + LR
    client_models = []
    for i, c in enumerate(clients):
        rf  = RandomForestClassifier(n_estimators=50, random_state=42).fit(c['X'], c['y'])
        xgb = XGBClassifier(n_estimators=50, use_label_encoder=False, eval_metric='logloss', random_state=42).fit(c['X'], c['y'])
        lr  = LogisticRegression(max_iter=500, random_state=42).fit(c['X'], c['y'])
        client_models.append({'rf': rf, 'xgb': xgb, 'lr': lr})
        print(f'  Client {i+1} trained')
    return client_models

print('Training tabular models (federated simulation)...')
tab_client_models = federated_tabular_train(X_tr, y_tr)

# ── Federated aggregation: average probabilities across clients ──
def aggregate_tabular_proba(client_models, X):
    all_probas = []
    for cm in client_models:
        p = (cm['rf'].predict_proba(X)[:,1] +
             cm['xgb'].predict_proba(X)[:,1] +
             cm['lr'].predict_proba(X)[:,1]) / 3
        all_probas.append(p)
    return np.mean(all_probas, axis=0)

tab_proba_te = aggregate_tabular_proba(tab_client_models, X_te)
tab_pred_te  = (tab_proba_te > 0.5).astype(int)
print('\nTabular Model Report:')
print(classification_report(y_te, tab_pred_te, target_names=['Real','Fake']))

# Save best client model (client 0 rf as representative)
joblib.dump(tab_client_models[0]['rf'],  'saved_models/tab_rf.pkl')
joblib.dump(tab_client_models[0]['xgb'], 'saved_models/tab_xgb.pkl')
joblib.dump(tab_client_models[0]['lr'],  'saved_models/tab_lr.pkl')
joblib.dump(scaler_tab, 'saved_models/scaler_tab.pkl')
print('Tabular models saved.')

Training tabular models (federated simulation)...
  Client 1 trained
  Client 2 trained

Tabular Model Report:
              precision    recall  f1-score   support

        Real       1.00      1.00      1.00       400
        Fake       1.00      0.99      0.99       271

    accuracy                           1.00       671
   macro avg       1.00      1.00      1.00       671
weighted avg       1.00      1.00      1.00       671

Tabular models saved.


In [6]:
import torch, pickle
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open('saved_models/vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden=128, num_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru   = nn.GRU(embed_dim, hidden, num_layers=num_layers, batch_first=True, dropout=0.3)
        self.drop  = nn.Dropout(0.4)
        self.fc    = nn.Linear(hidden, 1)
    def forward_logits(self, x):
        x = self.embed(x)
        _, h = self.gru(x)
        return self.fc(self.drop(h[-1])).squeeze(1)
    def forward(self, x):
        return torch.sigmoid(self.forward_logits(x))

gru_global = GRUClassifier(len(vocab)).to(DEVICE)
gru_global.load_state_dict(torch.load('saved_models/gru_global.pth', map_location=DEVICE))
gru_global.eval()
print('GRU loaded.')


GRU loaded.


## Section 4: Phase 1B — NLP Model (GRU) with Federated Simulation

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re, copy

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

# ── Tokenizer ─────────────────────────────────────────────────
def tokenize(text):
    return re.findall(r'\w+', text.lower())

all_tokens = [t for text in df_activity['content'] for t in tokenize(text)]
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in Counter(all_tokens).most_common(10000):
    vocab[word] = len(vocab)

MAX_LEN = 64

def encode(text, max_len=MAX_LEN):
    ids = [vocab.get(t, 1) for t in tokenize(text)][:max_len]
    return ids + [0] * (max_len - len(ids))

# ── Dataset ───────────────────────────────────────────────────
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.X = torch.tensor([encode(t) for t in texts], dtype=torch.long)
        self.y = torch.tensor(labels, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

# ── GRU Model ─────────────────────────────────────────────────
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden=128, num_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru   = nn.GRU(embed_dim, hidden, num_layers=num_layers, batch_first=True, dropout=0.3)
        self.fc    = nn.Linear(hidden, 1)
    def forward(self, x):
        x = self.embed(x)
        _, h = self.gru(x)
        return torch.sigmoid(self.fc(h[-1])).squeeze(1)

def train_gru(texts, labels, epochs=3, lr=1e-3):
    ds     = TextDataset(texts, labels)
    loader = DataLoader(ds, batch_size=64, shuffle=True)
    model  = GRUClassifier(len(vocab)).to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()
    for ep in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        print(f'    Epoch {ep+1}/{epochs} loss={total_loss/len(loader):.4f}')
    return model

# ── Federated simulation: 2 clients ───────────────────────────
texts  = df_activity['content'].tolist()
labels = df_activity['is_fake'].tolist()
mid    = len(texts) // 2

print('Training NLP GRU — Client 1...')
gru_c1 = train_gru(texts[:mid], labels[:mid])
print('Training NLP GRU — Client 2...')
gru_c2 = train_gru(texts[mid:], labels[mid:])

# ── FedAvg: average GRU weights ───────────────────────────────
gru_global = GRUClassifier(len(vocab)).to(DEVICE)
global_state = gru_global.state_dict()
for key in global_state:
    global_state[key] = (gru_c1.state_dict()[key].float() + gru_c2.state_dict()[key].float()) / 2
gru_global.load_state_dict(global_state)

torch.save(gru_global.state_dict(), 'saved_models/gru_global.pth')
import pickle
with open('saved_models/vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)
print('GRU global model saved.')

Using device: cpu
Training NLP GRU — Client 1...
    Epoch 1/3 loss=0.1860
    Epoch 2/3 loss=0.0001
    Epoch 3/3 loss=0.0000
Training NLP GRU — Client 2...
    Epoch 1/3 loss=0.1922
    Epoch 2/3 loss=0.0000
    Epoch 3/3 loss=0.0000
GRU global model saved.


In [26]:
# ==========================================
# FULL GRU PIPELINE (STABLE + COMPLETE)
# ==========================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import re, pickle
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

DEVICE = torch.device('cpu')
print("Device:", DEVICE)

# ==========================================
# LOAD DATA
# ==========================================

act = pd.read_csv('raw_user_activities 1(in).csv')

texts = act['content'].fillna('').tolist()[:50000]
labels = act['is_fake'].map({True:1, False:0, 'TRUE':1, 'FALSE':0}).fillna(0).astype(int).tolist()[:50000]

print("Dataset size:", len(texts))

# ==========================================
# TOKENIZER + VOCAB
# ==========================================

def tokenize(t):
    return re.findall(r'\w+', t.lower())

vocab = {'<PAD>':0, '<UNK>':1}

all_tokens = [tok for t in texts for tok in tokenize(t)]
for w, _ in Counter(all_tokens).most_common(5000):
    vocab[w] = len(vocab)

# Save vocab
with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

# ==========================================
# ENCODING
# ==========================================

def encode(text):
    ids = [vocab.get(t,1) for t in tokenize(text)][:30]
    return ids + [0]*(30-len(ids))

print("Encoding...")
X = []
for i, t in enumerate(texts):
    if i % 5000 == 0:
        print(f"Encoding {i}/{len(texts)}")
    X.append(encode(t))

X = torch.tensor(X)
y = torch.tensor(labels, dtype=torch.float32)

# ==========================================
# TRAIN / TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# MODEL
# ==========================================

class GRUModel(nn.Module):
    def __init__(self, vsz):
        super().__init__()
        self.emb = nn.Embedding(vsz, 32)
        self.gru = nn.GRU(32, 32, batch_first=True)
        self.fc  = nn.Linear(32,1)

    def forward(self,x):
        _, h = self.gru(self.emb(x))
        return torch.sigmoid(self.fc(h[-1])).squeeze()

model = GRUModel(len(vocab)).to(DEVICE)

opt = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

# ==========================================
# TRAINING
# ==========================================

print("\nTraining GRU...")

for ep in range(5):
    total = 0

    for i in range(0, len(X_train), 256):
        xb = X_train[i:i+256].to(DEVICE)
        yb = y_train[i:i+256].to(DEVICE)

        opt.zero_grad()
        out = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        opt.step()

        total += loss.item()

    avg_loss = total / (len(X_train)//256)
    print(f"Epoch {ep+1} Avg Loss: {avg_loss:.4f}")

# Save model
torch.save(model.state_dict(), "gru_model.pth")

print("\nTraining Done ✅")

# ==========================================
# EVALUATION
# ==========================================

model.eval()
preds = []

with torch.no_grad():
    for i in range(0, len(X_test), 256):
        xb = X_test[i:i+256].to(DEVICE)
        p = (model(xb) > 0.5).int().cpu().numpy()
        preds.extend(p)

print("\nEvaluation Report:")
print(classification_report(y_test, preds, target_names=['Real','Fake']))

# ==========================================
# DONE
# ==========================================

print("\nALL DONE 🚀")

Device: cpu
Dataset size: 50000
Encoding...
Encoding 0/50000
Encoding 5000/50000
Encoding 10000/50000
Encoding 15000/50000
Encoding 20000/50000
Encoding 25000/50000
Encoding 30000/50000
Encoding 35000/50000
Encoding 40000/50000
Encoding 45000/50000

Training GRU...
Epoch 1 Avg Loss: 0.3038
Epoch 2 Avg Loss: 0.0064
Epoch 3 Avg Loss: 0.0023
Epoch 4 Avg Loss: 0.0013
Epoch 5 Avg Loss: 0.0009

Training Done ✅

Evaluation Report:
              precision    recall  f1-score   support

        Real       1.00      1.00      1.00      7772
        Fake       1.00      1.00      1.00      2228

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000


ALL DONE 🚀


## Section 5: Phase 1C — Vision Model (ResNet18) with Federated Simulation

In [27]:
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import requests
from io import BytesIO
import os

IMG_DIR = 'profile_images'
os.makedirs(IMG_DIR, exist_ok=True)

transform = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

def download_image(url, path, timeout=5):
    try:
        r = requests.get(url, timeout=timeout)
        img = Image.open(BytesIO(r.content)).convert('RGB')
        img.save(path)
        return True
    except:
        return False

# Download up to 500 images (for prototype speed)
df_vis = df_profile[['id','profile_image_url','is_fake']].dropna(subset=['profile_image_url']).head(500).copy()
df_vis['img_path'] = df_vis['id'].apply(lambda x: os.path.join(IMG_DIR, f'{x}.jpg'))

print('Downloading profile images (up to 500)...')
downloaded = []
for _, row in df_vis.iterrows():
    if not os.path.exists(row['img_path']):
        ok = download_image(row['profile_image_url'], row['img_path'])
    else:
        ok = True
    downloaded.append(ok)
df_vis['downloaded'] = downloaded
df_vis = df_vis[df_vis['downloaded']]
print(f'Downloaded {len(df_vis)} images')

Downloaded 0 images


In [28]:
class ProfileImageDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert('RGB')
        except:
            img = Image.new('RGB', (64,64))
        return self.transform(img), torch.tensor(self.labels[i], dtype=torch.float32)

def train_resnet(paths, labels, epochs=3):
    ds     = ProfileImageDataset(paths, labels, transform)
    loader = DataLoader(ds, batch_size=32, shuffle=True)
    model  = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 1)
    model  = model.to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()
    for ep in range(epochs):
        model.train()
        total = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb).squeeze(1), yb)
            loss.backward()
            opt.step()
            total += loss.item()
        print(f'    Epoch {ep+1}/{epochs} loss={total/len(loader):.4f}')
    return model

paths  = df_vis['img_path'].tolist()
labels = df_vis['is_fake'].tolist()
mid    = len(paths) // 2

if len(paths) >= 10:
    print('Training Vision ResNet18 — Client 1...')
    res_c1 = train_resnet(paths[:mid], labels[:mid])
    print('Training Vision ResNet18 — Client 2...')
    res_c2 = train_resnet(paths[mid:], labels[mid:])

    # FedAvg
    res_global = models.resnet18(weights=None)
    res_global.fc = nn.Linear(res_global.fc.in_features, 1)
    res_global = res_global.to(DEVICE)
    g_state = res_global.state_dict()
    for key in g_state:
        g_state[key] = (res_c1.state_dict()[key].float() + res_c2.state_dict()[key].float()) / 2
    res_global.load_state_dict(g_state)
    torch.save(res_global.state_dict(), 'saved_models/resnet_global.pth')
    print('ResNet18 global model saved.')
    VISION_AVAILABLE = True
else:
    print('Not enough images downloaded — vision model skipped.')
    VISION_AVAILABLE = False

Not enough images downloaded — vision model skipped.


## Section 6: Federated Aggregator Summary

In [29]:
print('=== Federated Aggregator Summary ===')
print('Tabular  (RF/XGB/LR) — FedAvg: averaged probabilities across 2 client nodes')
print('NLP      (GRU)        — FedAvg: averaged model weights across 2 client nodes')
print('Vision   (ResNet18)   — FedAvg: averaged model weights across 2 client nodes')
print('Global hybrid models saved to saved_models/')

=== Federated Aggregator Summary ===
Tabular  (RF/XGB/LR) — FedAvg: averaged probabilities across 2 client nodes
NLP      (GRU)        — FedAvg: averaged model weights across 2 client nodes
Vision   (ResNet18)   — FedAvg: averaged model weights across 2 client nodes
Global hybrid models saved to saved_models/


## Section 7: Phase 2 — Meta-Learner (Stacking)

In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np

# ── Build meta-features from all 3 models on activity test set ──
# We use activity dataset as common ground (has labels + text + tabular)

act_tab_cols = ['likes','comments','shares','hour_of_day','day_of_week','is_weekend',
                'character_count','hashtag_count','mention_count','contains_url',
                'device','platform','media_type','content_type','language']

X_act = df_activity[act_tab_cols].fillna(0).values
y_act = df_activity['is_fake'].values
texts_act = df_activity['content'].tolist()

scaler_act = StandardScaler()
X_act_sc   = scaler_act.fit_transform(X_act)
joblib.dump(scaler_act, 'saved_models/scaler_act.pkl')

Xtr_a, Xte_a, ytr_a, yte_a, ttr_a, tte_a = train_test_split(
    X_act_sc, y_act, texts_act, test_size=0.2, random_state=42)

# Train a fresh tabular model on activity data
rf_act = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xtr_a, ytr_a)
joblib.dump(rf_act, 'saved_models/rf_act.pkl')
p_tab = rf_act.predict_proba(Xte_a)[:,1]

# GRU probabilities
gru_global.eval()
te_ds2 = TextDataset(tte_a, yte_a)
te_ld2 = DataLoader(te_ds2, batch_size=64)
p_nlp  = []
with torch.no_grad():
    for xb, yb in te_ld2:
        p_nlp.extend(gru_global(xb.to(DEVICE)).cpu().numpy())
p_nlp = np.array(p_nlp)

# Stack meta-features
if VISION_AVAILABLE:
    # Use 0.5 as placeholder vision proba for activity data (no images)
    p_vis = np.full(len(p_tab), 0.5)
    meta_X_te = np.column_stack([p_tab, p_nlp, p_vis])
else:
    meta_X_te = np.column_stack([p_tab, p_nlp])

# Train meta-learner on train split
p_tab_tr = rf_act.predict_proba(Xtr_a)[:,1]
tr_ds2   = TextDataset(ttr_a, ytr_a)
tr_ld2   = DataLoader(tr_ds2, batch_size=64)
p_nlp_tr = []
with torch.no_grad():
    for xb, yb in tr_ld2:
        p_nlp_tr.extend(gru_global(xb.to(DEVICE)).cpu().numpy())
p_nlp_tr = np.array(p_nlp_tr)

if VISION_AVAILABLE:
    p_vis_tr  = np.full(len(p_tab_tr), 0.5)
    meta_X_tr = np.column_stack([p_tab_tr, p_nlp_tr, p_vis_tr])
else:
    meta_X_tr = np.column_stack([p_tab_tr, p_nlp_tr])

meta_learner = LogisticRegression(max_iter=500).fit(meta_X_tr, ytr_a)
joblib.dump(meta_learner, 'saved_models/meta_learner.pkl')

meta_pred = meta_learner.predict(meta_X_te)
print('Meta-Learner Report:')
print(classification_report(yte_a, meta_pred, target_names=['Real','Fake']))

Meta-Learner Report:
              precision    recall  f1-score   support

        Real       1.00      1.00      1.00     20211
        Fake       1.00      1.00      1.00      6046

    accuracy                           1.00     26257
   macro avg       1.00      1.00      1.00     26257
weighted avg       1.00      1.00      1.00     26257



## Section 8: Phase 3 — FastAPI Server

In [31]:
api_code = '''
from fastapi import FastAPI, UploadFile, File
from pydantic import BaseModel
from typing import Optional
import numpy as np
import torch, pickle, joblib, re
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from io import BytesIO

app = FastAPI(title="Fake Profile Detection API")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 64

# ── Load artifacts ────────────────────────────────────────────
rf_act       = joblib.load("saved_models/rf_act.pkl")
scaler_act   = joblib.load("saved_models/scaler_act.pkl")
meta_learner = joblib.load("saved_models/meta_learner.pkl")
with open("saved_models/vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden=128, num_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru   = nn.GRU(embed_dim, hidden, num_layers=num_layers, batch_first=True, dropout=0.3)
        self.fc    = nn.Linear(hidden, 1)
    def forward(self, x):
        x = self.embed(x)
        _, h = self.gru(x)
        return torch.sigmoid(self.fc(h[-1])).squeeze(1)

gru = GRUClassifier(len(vocab)).to(DEVICE)
gru.load_state_dict(torch.load("saved_models/gru_global.pth", map_location=DEVICE))
gru.eval()

try:
    resnet = models.resnet18(weights=None)
    resnet.fc = nn.Linear(resnet.fc.in_features, 1)
    resnet.load_state_dict(torch.load("saved_models/resnet_global.pth", map_location=DEVICE))
    resnet = resnet.to(DEVICE).eval()
    VISION_AVAILABLE = True
except:
    VISION_AVAILABLE = False

transform = T.Compose([
    T.Resize((64,64)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

def tokenize(text): return re.findall(r"\\w+", text.lower())
def encode(text):
    ids = [vocab.get(t,1) for t in tokenize(text)][:MAX_LEN]
    return ids + [0]*(MAX_LEN - len(ids))

# ── Schemas ───────────────────────────────────────────────────
class TabularInput(BaseModel):
    likes: float = 0; comments: float = 0; shares: float = 0
    hour_of_day: float = 12; day_of_week: float = 1; is_weekend: float = 0
    character_count: float = 0; hashtag_count: float = 0
    mention_count: float = 0; contains_url: float = 0
    device: float = 0; platform: float = 0; media_type: float = 0
    content_type: float = 0; language: float = 0

class TextInput(BaseModel):
    text: str

class ProfileInput(BaseModel):
    likes: float = 0; comments: float = 0; shares: float = 0
    hour_of_day: float = 12; day_of_week: float = 1; is_weekend: float = 0
    character_count: float = 0; hashtag_count: float = 0
    mention_count: float = 0; contains_url: float = 0
    device: float = 0; platform: float = 0; media_type: float = 0
    content_type: float = 0; language: float = 0
    text: str = ""

# ── Endpoints ─────────────────────────────────────────────────
@app.post("/predict/tabular")
def predict_tabular(inp: TabularInput):
    x = np.array([[inp.likes, inp.comments, inp.shares, inp.hour_of_day,
                   inp.day_of_week, inp.is_weekend, inp.character_count,
                   inp.hashtag_count, inp.mention_count, inp.contains_url,
                   inp.device, inp.platform, inp.media_type,
                   inp.content_type, inp.language]])
    x_sc = scaler_act.transform(x)
    prob = float(rf_act.predict_proba(x_sc)[0,1])
    return {"probability": prob, "prediction": "Fake" if prob > 0.5 else "Real"}

@app.post("/predict/text")
def predict_text(inp: TextInput):
    ids = torch.tensor([encode(inp.text)], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        prob = float(gru(ids).cpu().numpy()[0])
    return {"probability": prob, "prediction": "Fake" if prob > 0.5 else "Real"}

@app.post("/predict/vision")
async def predict_vision(file: UploadFile = File(...)):
    if not VISION_AVAILABLE:
        return {"probability": 0.5, "prediction": "Unknown", "note": "Vision model not available"}
    img = Image.open(BytesIO(await file.read())).convert("RGB")
    x   = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = float(torch.sigmoid(resnet(x)).cpu().numpy()[0][0])
    return {"probability": prob, "prediction": "Fake" if prob > 0.5 else "Real"}

@app.post("/predict/full")
def predict_full(inp: ProfileInput):
    # Tabular proba
    x = np.array([[inp.likes, inp.comments, inp.shares, inp.hour_of_day,
                   inp.day_of_week, inp.is_weekend, inp.character_count,
                   inp.hashtag_count, inp.mention_count, inp.contains_url,
                   inp.device, inp.platform, inp.media_type,
                   inp.content_type, inp.language]])
    p_tab = float(rf_act.predict_proba(scaler_act.transform(x))[0,1])
    # NLP proba
    ids   = torch.tensor([encode(inp.text)], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        p_nlp = float(gru(ids).cpu().numpy()[0])
    # Vision placeholder
    p_vis = 0.5
    meta_x = np.array([[p_tab, p_nlp, p_vis]]) if VISION_AVAILABLE else np.array([[p_tab, p_nlp]])
    prob   = float(meta_learner.predict_proba(meta_x)[0,1])
    label  = "Bot/Fake" if prob > 0.5 else "Human/Real"
    return {
        "final_prediction": label,
        "confidence": round(prob, 4),
        "probabilities": {"tabular": round(p_tab,4), "nlp": round(p_nlp,4), "vision": round(p_vis,4)}
    }

@app.get("/health")
def health(): return {"status": "ok", "vision_available": VISION_AVAILABLE}
'''

with open('api.py', 'w') as f:
    f.write(api_code.strip())
print('api.py written.')

api.py written.


## Section 9: Start FastAPI Server & Test

In [32]:
import subprocess, time, requests as req

# Start server in background
server = subprocess.Popen(['python', '-m', 'uvicorn', 'api:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(4)
print('Server started on http://localhost:8000')
print('Docs at: http://localhost:8000/docs')

INFO:     Started server process [5866]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 48] error while attempting to bind on address ('0.0.0.0', 8000): [errno 48] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Server started on http://localhost:8000
Docs at: http://localhost:8000/docs


In [33]:
import sys
!{sys.executable} -m pip install python-multipart




In [34]:
import requests as req, json

BASE = 'http://localhost:8000'

# Test /health
print('Health:', req.get(f'{BASE}/health').json())

# Test /predict/text
r = req.post(f'{BASE}/predict/text', json={'text': 'Buy followers now! Click here for free likes!!!'})
print('Text (spam):', r.json())

r = req.post(f'{BASE}/predict/text', json={'text': 'Had a great morning run today, feeling refreshed!'})
print('Text (normal):', r.json())

# Test /predict/tabular
r = req.post(f'{BASE}/predict/tabular', json={
    'likes': 0, 'comments': 0, 'shares': 0, 'hour_of_day': 3,
    'day_of_week': 1, 'is_weekend': 0, 'character_count': 5,
    'hashtag_count': 10, 'mention_count': 8, 'contains_url': 1,
    'device': 0, 'platform': 0, 'media_type': 0, 'content_type': 0, 'language': 0
})
print('Tabular (suspicious):', r.json())

# Test /predict/full (meta-learner)
r = req.post(f'{BASE}/predict/full', json={
    'likes': 0, 'comments': 0, 'shares': 0, 'hour_of_day': 3,
    'day_of_week': 1, 'is_weekend': 0, 'character_count': 5,
    'hashtag_count': 10, 'mention_count': 8, 'contains_url': 1,
    'device': 0, 'platform': 0, 'media_type': 0, 'content_type': 0, 'language': 0,
    'text': 'Follow me follow back free followers click link'
})
print('Full prediction:', json.dumps(r.json(), indent=2))

INFO:     127.0.0.1:49816 - "GET /health HTTP/1.1" 200 OK
Health: {'status': 'ok', 'vision_available': False}
INFO:     127.0.0.1:49818 - "POST /predict/text HTTP/1.1" 200 OK
Text (spam): {'probability': 0.0727815255522728, 'prediction': 'Real'}
INFO:     127.0.0.1:49821 - "POST /predict/text HTTP/1.1" 200 OK
Text (normal): {'probability': 0.07257559150457382, 'prediction': 'Real'}
INFO:     127.0.0.1:49823 - "POST /predict/tabular HTTP/1.1" 200 OK
Tabular (suspicious): {'probability': 0.31, 'prediction': 'Real'}
INFO:     127.0.0.1:49825 - "POST /predict/full HTTP/1.1" 200 OK
Full prediction: {
  "final_prediction": "Human/Real",
  "confidence": 0.0368,
  "probabilities": {
    "tabular": 0.31,
    "nlp": 0.0728,
    "vision": 0.5
  }
}
